In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import zipfile
import io
from pyspark.sql import functions as F
from pathlib import Path

In [ ]:
spark = SparkSession.builder.appName("Exercise_7").getOrCreate()

In [ ]:
def read_csv_from_zip_to_df(spark, zip_path):

    df_list = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.namelist():
            if member.startswith("__MACOSX") or not member.lower().endswith(".csv"):
                continue
            print(f"Reading file: {member} from {zip_path}")
            raw = z.read(member)                         
            text = raw.decode("utf-8", errors="ignore") 
            
            lines = [line for line in text.splitlines() if line.strip() != ""]
            rdd = spark.sparkContext.parallelize(lines)
            df = spark.read.csv(rdd, header=True, inferSchema=True)
            df_list.append(df)

    if not df_list:
        return None
  
    out = df_list[0]
    for other in df_list[1:]:
        out = out.unionByName(other, allowMissingColumns=True)
    return out

In [ ]:

zip_files = [
    "data/hard-drive-2022-01-01-failures.csv.zip"
    ]

dfs = {}
for z in zip_files:
    df = read_csv_from_zip_to_df(spark, z)
    if df is None:
        print(f"No CSV files found in {z}")
    else:
        dfs[z] = df

df = dfs.get("data/hard-drive-2022-01-01-failures.csv.zip")

if df is None:
    print("df1 not found — exiting")
    # spark.stop()

# show a quick preview
df.show(5, truncate=False)
print("Files Downloaded Successfully!11")

In [ ]:
df.columns
df.printSchema()
len(df.columns)

In [ ]:
# path = "data/hard-drive-2022-01-01-failures.csv.zip"
# file_path = Path(path)
# file_name = Path(file_path.stem).stem

# df = df.withColumn("source_file", F.lit(file_name))
# result = df.select("source_file")

# result.coalesce(1).write.csv("Results/adding_column_and_ingesting_file_name", header=True, mode="overwrite")


In [ ]:
def adding_column_and_ingesting_file_name(df):
    path = "data/hard-drive-2022-01-01-failures.csv.zip"
    file_path = Path(path)
    file_name = Path(file_path.stem).stem
    
    df = df.withColumn("source_file", F.lit(file_name))
    result = df.select("source_file")
    
    result.coalesce(1).write.csv("Results/adding_column_and_ingesting_file_name", header=True, mode="overwrite")
    return df.select("source_file").show()
adding_column_and_ingesting_file_name(df)

In [ ]:
df['sour']

In [ ]:
len(df.columns)
df.select("source_file").show(10)

In [ ]:
df['so']

In [ ]:
# Q2. Pull the date located inside the string of the source_file column. 
# Final data-type must be date or timestamp, not a string. Call the new column file_date.

In [ ]:
a = df.columns

In [ ]:
type(a)

In [ ]:
len(a)

In [ ]:
path = "data/hard-drive-2022-01-01-failures.csv.zip"
    file_path = Path(path)
    file_name = Path(file_path.stem).stem
    
    df = df.withColumn("source_file", F.lit(file_name))
    result = df.select("source_file")
    
    result.coalesce(1).write.csv("Results/adding_column_and_ingesting_file_name", header=True, mode="overwrite")

In [ ]:
df.select("source_file")

# =======================================

In [ ]:
df = df.withColumnRenamed("date", "file_date")

In [ ]:
df.columns

In [ ]:
df.withColumn("file_date", to_date("source_file")).show()

In [ ]:
# Q2.Pull the date located inside the string of the source_file column. 
# Final data-type must be date or timestamp, not a string. Call the new column file_date
def add_date_column_using_source_files_column_data(df):

    path = "data/hard-drive-2022-01-01-failures.csv.zip"
    file_path = Path(path)
    file_name = Path(file_path.stem).stem
    
    df = df.withColumn("source_file", F.lit(file_name))
    
    df = df.withColumn(
        "file_date",
        to_date(regexp_extract(F.col(("source_file")), r'(\d{4}-\d{2}-\d{2})', 1))
    )
    result = df.select("file_date")
    result.coalesce(1).write.csv("Results/add_date_column_using_source_files_column_data", header=True, mode="overwrite")
    return result.show()

add_date_column_using_source_files_column_data(df)

In [ ]:
len(df_date.columns)
df_date.printSchema()

In [ ]:
df.select("file_date").show()

In [ ]:
df.columns

In [ ]:
# Q3. Add a new column called brand. It will be based on the column model. If the column model has a space ... aka   in it, 
# split on that space. The value found before the space   will be considered the brand. 
# If there is no space to split on, fill in a value called unknown for the brand

In [ ]:
def brand_name(df):
    df_split = df.withColumn("model_extracts", split(col("model"), " "))
    
    df_result = df_split.withColumn(
    "brand", when(size(col("model_extracts"))>1, col("model_extracts").getItem(0)).otherwise("Unknown"))
    
    df_result = df_result.drop("model_extracts")
    result = df_result.select("brand")
    
    result.coalesce(1).write.csv("Results/brand_name", header=True, mode="overwrite")
    return result.show(10, truncate=True)

brand_name(df)

In [ ]:
# Inspect a column called capacity_bytes. Create a secondary DataFrame that relates capacity_bytes to 
# the model column, create "buckets" / "rankings" for those models with the most capacity to the least. 
# Bring back that data as a column called storage_ranking into the main dataset.
df.columns

In [195]:
df.select("capacity_bytes").show()
dff = df

+--------------+
|capacity_bytes|
+--------------+
|14000519643136|
|12000138625024|
| 8001563222016|
| 8001563222016|
|14000519643136|
| 4000787030016|
| 8001563222016|
|14000519643136|
|14000519643136|
|12000138625024|
|12000138625024|
|12000138625024|
|12000138625024|
|14000519643136|
|12000138625024|
|12000138625024|
|12000138625024|
|14000519643136|
|12000138625024|
|12000138625024|
+--------------+
only showing top 20 rows


25/11/07 15:21:12 WARN TaskSetManager: Stage 45 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [200]:
df_split = df.withColumn("model_extracts", split(col("model"), " "))
df_split = df_split.withColumn("brands", when(size(col("model_extracts"))>1, col("model_extracts").getItem(0)).otherwise("NULL"))

In [201]:
df_split.select("capacity_bytes", "brands").show(10)

+--------------+-------+
|capacity_bytes| brands|
+--------------+-------+
|14000519643136|   NULL|
|12000138625024|   NULL|
| 8001563222016|   NULL|
| 8001563222016|   NULL|
|14000519643136|TOSHIBA|
| 4000787030016|   HGST|
| 8001563222016|   NULL|
|14000519643136|TOSHIBA|
|14000519643136|TOSHIBA|
|12000138625024|   NULL|
+--------------+-------+
only showing top 10 rows


25/11/07 15:25:18 WARN TaskSetManager: Stage 50 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [213]:
window_spec = Window.partitionBy("brands").orderBy(col("capacity_bytes").desc())

df_with_rank = df_split.filter(F.col("brands").isNotNull()).withColumn("Brand_wise_Ranking", dense_rank().over(window_spec))



In [214]:
df_with_rank.select("brands","Brand_wise_Ranking").groupBy("Brand_wise_Ranking")agg(count("brands").alias("Number_of")).groupBy("").show()

SyntaxError: invalid syntax (3030613290.py, line 1)

In [220]:
df_with_rank.select("brands", "Brand_wise_Ranking").orderBy("Brand_wise_Ranking").distinct().show()

25/11/07 15:40:25 WARN TaskSetManager: Stage 97 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
[Stage 97:>                                                         (0 + 4) / 4]

+-----------+------------------+
|     brands|Brand_wise_Ranking|
+-----------+------------------+
|   DELLBOSS|                 1|
|       HGST|                 1|
|       HGST|                 2|
|       HGST|                 3|
|       HGST|                 4|
|    Hitachi|                 1|
|       NULL|                 1|
|       NULL|                 2|
|       NULL|                 3|
|       NULL|                 4|
|       NULL|                 5|
|       NULL|                 6|
|       NULL|                 7|
|       NULL|                 8|
|       NULL|                 9|
|       NULL|                10|
|       NULL|                11|
|       NULL|                12|
|ST1000LM024|                 1|
| ST500LM012|                 1|
+-----------+------------------+
only showing top 20 rows


In [217]:
df_with_rank.select("Brand_wise_Ranking").distinct().count()

25/11/07 15:37:23 WARN TaskSetManager: Stage 87 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

12

In [224]:
df_with_rank.groupBy("Brand_wise_Ranking").agg(count("Brand_wise_Ranking")).orderBy("Brand_wise_Ranking").show()

25/11/07 15:43:09 WARN TaskSetManager: Stage 109 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
[Stage 109:==============>                                          (1 + 3) / 4]

+------------------+-------------------------+
|Brand_wise_Ranking|count(Brand_wise_Ranking)|
+------------------+-------------------------+
|                 1|                    37466|
|                 2|                    58006|
|                 3|                    15287|
|                 4|                    50163|
|                 5|                     1337|
|                 6|                    24693|
|                 7|                      891|
|                 8|                    18649|
|                 9|                      265|
|                10|                       80|
|                11|                       90|
|                12|                       27|
+------------------+-------------------------+



In [229]:
# Inspect a column called capacity_bytes. Create a secondary DataFrame that relates capacity_bytes to the model column,
# create "buckets" / "rankings" for those models with the most capacity to the least. Bring back that data as a column 
# called storage_ranking into the main dataset.
def  brand_wise_ranking(df):
    df_split = df.withColumn("model_extracts", split(col("model"), " "))
    df_split = df_split.withColumn("brands", when(size(col("model_extracts"))>1, col("model_extracts").getItem(0)).otherwise("NULL"))
    
    window_spec = Window.partitionBy("brands").orderBy(col("capacity_bytes").desc())
    
    df_with_rank = df_split.filter(F.col("brands").isNotNull()).withColumn("Brand_wise_Ranking", dense_rank().over(window_spec))
    
    
    result = df_with_rank.select("brands", "Brand_wise_Ranking")
    
    result.coalesce(1).write.csv("Results/Ranking", header=True, mode="overwrite")
    
    return result.show()

brand_wise_ranking(df)

25/11/07 15:49:18 WARN TaskSetManager: Stage 124 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
25/11/07 15:49:20 WARN TaskSetManager: Stage 127 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
[Stage 127:==============>                                          (1 + 3) / 4]

+--------+------------------+
|  brands|Brand_wise_Ranking|
+--------+------------------+
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
|DELLBOSS|                 1|
+--------+------------------+
only showing top 20 rows


In [227]:
df_with_rank.select("brands", "Brand_wise_Ranking").distinct()

25/11/07 15:46:58 WARN TaskSetManager: Stage 118 contains a task of very large size (17793 KiB). The maximum recommended task size is 1000 KiB.
[Stage 118:==========================================>              (3 + 1) / 4]

+-----------+------------------+
|     brands|Brand_wise_Ranking|
+-----------+------------------+
|   DELLBOSS|                 1|
|       HGST|                 1|
|       HGST|                 2|
|       HGST|                 3|
|       HGST|                 4|
|    Hitachi|                 1|
|       NULL|                 1|
|       NULL|                 2|
|       NULL|                 3|
|       NULL|                 4|
|       NULL|                 5|
|       NULL|                 6|
|       NULL|                 7|
|       NULL|                 8|
|       NULL|                 9|
|       NULL|                10|
|       NULL|                11|
|       NULL|                12|
|ST1000LM024|                 1|
| ST500LM012|                 1|
+-----------+------------------+
only showing top 20 rows


In [232]:
updated_df = df.withColumn("Rankings", F.dense_rank().over(window_spec))

{"ts": "2025-11-07 15:55:17.240", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `brands` cannot be resolved. Did you mean one of the following? [`date`, `model`, `failure`, `serial_number`, `smart_1_raw`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor87.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1406.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `brands` cannot be resolved. Did you mean one of the following? [`date`, `model`, `failure`, `serial_number`, `smart_1_raw`]. SQLSTATE: 42703;\n'Project [date#8344, serial_number#8345, model#8346, capacity_bytes#8347L, failure#8348, smart_1_normalized#8349, smar

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `brands` cannot be resolved. Did you mean one of the following? [`date`, `model`, `failure`, `serial_number`, `smart_1_raw`]. SQLSTATE: 42703;
'Project [date#8344, serial_number#8345, model#8346, capacity_bytes#8347L, failure#8348, smart_1_normalized#8349, smart_1_raw#8350, smart_2_normalized#8351, smart_2_raw#8352, smart_3_normalized#8353, smart_3_raw#8354, smart_4_normalized#8355, smart_4_raw#8356, smart_5_normalized#8357, smart_5_raw#8358, smart_7_normalized#8359, smart_7_raw#8360L, smart_8_normalized#8361, smart_8_raw#8362, smart_9_normalized#8363, smart_9_raw#8364, smart_10_normalized#8365, smart_10_raw#8366, smart_11_normalized#8367, smart_11_raw#8368, ... 156 more fields]
+- Project [date#8344, serial_number#8345, model#8346, capacity_bytes#8347L, failure#8348, smart_1_normalized#8349, smart_1_raw#8350, smart_2_normalized#8351, smart_2_raw#8352, smart_3_normalized#8353, sma

In [234]:
len(df_with_rank.columns)

183

In [ ]:
# Q5. Create a column called primary_key that is hash of columns that make a record umique in this dataset.